# Norms_Train — 우리 WFLW 모델로 기준표(norms.json) + 얼굴형 MLP 재생성 (방법 A)

MediaPipe 판 `FaceReading_Train.ipynb` 와 같은 일을 하되, 점 찾기를 **우리가 학습한 `wflw_mesh.tflite`** 로 합니다.
기준표는 "같은 측정기로 잰 값"이어야 하므로 보드가 우리 모델을 쓰면 기준표도 우리 모델로 만들어야 합니다.

**만드는 것 (보드로 복사)**: `norms.json`(42개 측정값 분위수), `scaler.json`, `faceshape_mlp.tflite`, `metrics.csv`

**미리 준비할 것**
- `WFLW_Train.ipynb` 를 먼저 끝내고 `save/wflw_mesh.tflite`, `save/face_detection_yunet_2023mar.onnx` 가 있을 것
- Kaggle Face Shape Dataset (niten19) zip → 드라이브 `ai_semiconductor/Python/examples/files/FaceShape.zip`
- 드라이브 `proj_face/FaceReading_3/` 에 `wflw_prep.py`, `wflw_mesh.py`, `face_metrics.py`, `hairline.py` (이 노트북과 같은 폴더)


## 1. 드라이브 · 경로 · 코드

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive'); colab = True
except Exception:
    colab = False

In [ ]:
import os, sys, shutil, glob, json, time
if colab:
    BASE = '/content/drive/MyDrive/ai_semiconductor/Python/examples/'
    CODE_DIR = BASE + 'proj_face/FaceReading_3/'; save_dir = BASE + 'files/save/'
    dataset_zip = BASE + 'files/FaceShape.zip'; work_dir = '/content/faceshape'
else:
    CODE_DIR = './'; save_dir = '../files/save/'; dataset_zip = '../files/FaceShape.zip'; work_dir = './faceshape'
os.makedirs(work_dir, exist_ok=True); os.makedirs(save_dir, exist_ok=True)
# 코드와 모델을 작업 폴더로 모아서 import (wflw_mesh 는 같은 폴더의 모델 파일을 찾음)
for f in ['wflw_prep.py', 'wflw_mesh.py', 'face_metrics.py', 'hairline.py']:
    shutil.copy(CODE_DIR + f, '/content/' if colab else './')
for f in ['wflw_mesh.tflite', 'face_detection_yunet_2023mar.onnx']:
    src = save_dir + f if os.path.exists(save_dir + f) else CODE_DIR + f
    shutil.copy(src, '/content/' if colab else './')
if colab: os.chdir('/content')
sys.path.insert(0, '.')
import numpy as np, cv2
from face_metrics import *
from wflw_mesh import WflwMesh
print(len(METRIC_NAMES), '개 측정값')
mesh = WflwMesh()

## 2. 데이터 압축 해제

In [ ]:
!unzip -q -o {dataset_zip} -d {work_dir}
def find_split(name):
    hits = glob.glob(os.path.join(work_dir, '**', name), recursive=True); return hits[0] if hits else None
train_root, test_root = find_split('training_set'), find_split('testing_set')
CLASSES = ['Heart', 'Oblong', 'Oval', 'Round', 'Square']
def class_dirs(root):
    return {c: os.path.join(root, d) for c in CLASSES for d in os.listdir(root) if d.lower() == c.lower()}
train_dirs, test_dirs = class_dirs(train_root), class_dirs(test_root)
for c in CLASSES: print(c, 'train', len(os.listdir(train_dirs[c])), 'test', len(os.listdir(test_dirs[c])))

## 3. 헤어라인 (선택, 보드 설정과 같게)
Face Mesh 와 마찬가지로 우리 모델도 이마 위 점이 없어 `wflw_prep.add_virtual_points` 가 미간에서 코 길이만큼 위를 이마 상단으로 잡습니다.
분할 모델이 있으면 실제 헤어라인으로 바꿔 잽니다. **보드에 `selfie_multiclass_256x256.tflite` 를 둘 거면 여기서도 True.**

In [ ]:
USE_HAIRLINE = True
segmenter = None
if USE_HAIRLINE:
    !pip -q install mediapipe
    import urllib.request
    from hairline import Segmenter, hairline_from_mask, MODEL_FILE
    if not os.path.exists(MODEL_FILE):
        urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_multiclass_256x256/float32/latest/' + MODEL_FILE, MODEL_FILE)
    segmenter = Segmenter(MODEL_FILE)
    if segmenter.ok: shutil.copy(MODEL_FILE, save_dir)
    else: segmenter = None
print('헤어라인', '사용' if segmenter else '미사용')

## 4. 모든 사진 측정 → metrics.csv  (5,000장, 5~10분)

In [ ]:
import pandas as pd
rows = []; t0 = time.time(); n_hl = 0; n_noface = 0
for split, dirs in (('train', train_dirs), ('test', test_dirs)):
    for ci, c in enumerate(CLASSES):
        files = sorted(glob.glob(os.path.join(dirs[c], '*'))); ok = 0
        for f in files:
            img = cv2.imread(f)
            if img is None: continue
            if max(img.shape[:2]) > 800:
                s = 800 / max(img.shape[:2]); img = cv2.resize(img, None, fx=s, fy=s)
            pts = mesh(img)
            if pts is None: n_noface += 1; continue
            yaw, roll = head_pose(pts)
            if abs(yaw) > 0.2 or abs(roll) > 15: continue          # 정면이 아니면 제외
            hl = None
            if segmenter is not None:
                mk = segmenter.mask(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                if mk is not None: hl = hairline_from_mask(mk, pts, P)
            n_hl += hl is not None
            m = compute_metrics(pts, hl); m.update(split=split, label=ci, cls=c, file=os.path.basename(f)); rows.append(m); ok += 1
        print(f'{split:5s} {c:7s} {ok:4d}/{len(files)} 장 사용  ({time.time()-t0:.0f}s)')
df = pd.DataFrame(rows)
print('얼굴 못 찾음:', n_noface, '| 헤어라인 적용:', n_hl, '/', len(df))
df.to_csv(save_dir + 'metrics.csv', index=False)
df[METRIC_NAMES].describe().T[['mean', 'std', 'min', 'max']].round(3)

## 5. norms.json (측정값마다 0~100 백분위 분위수 101개)

In [ ]:
q = {k: np.percentile(df[k].values, np.arange(101)).round(5).tolist() for k in METRIC_NAMES}
norms = dict(source='Kaggle Face Shape Dataset (niten19) + our WFLW model (wflw_mesh.tflite)', n=int(len(df)),
             metric_names=METRIC_NAMES, quantiles=q, hairline=bool(segmenter))
json.dump(norms, open(save_dir + 'norms.json', 'w'), indent=0)
print('norms.json 저장 —', len(df), '명 기준,', len(METRIC_NAMES), '개 측정값')
df.groupby('cls')[['face_ratio', 'forehead_w', 'cheek_w', 'jaw_w', 'jaw_angle']].mean().round(3)

## 6. 얼굴형 MLP (측정값 42개 → 5클래스)

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
tr, te = df[df.split == 'train'], df[df.split == 'test']
X_tr, y_tr = tr[METRIC_NAMES].values.astype(np.float32), tr['label'].values
X_te, y_te = te[METRIC_NAMES].values.astype(np.float32), te['label'].values
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_te = scaler.transform(X_tr).astype(np.float32), scaler.transform(X_te).astype(np.float32)
json.dump(dict(mean=scaler.mean_.tolist(), std=scaler.scale_.tolist(), metric_names=METRIC_NAMES), open(save_dir + 'scaler.json', 'w'))

model = tf.keras.Sequential([
    tf.keras.Input(shape=(len(METRIC_NAMES),)),
    tf.keras.layers.Dense(64, activation='relu'), tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'), tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(5, activation='softmax')])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(Xs_tr, y_tr, epochs=150, batch_size=64, validation_data=(Xs_te, y_te), verbose=0,
                    callbacks=[tf.keras.callbacks.EarlyStopping(patience=25, restore_best_weights=True)])
print('test accuracy: %.3f' % model.evaluate(Xs_te, y_te, verbose=0)[1])
from sklearn.metrics import confusion_matrix, classification_report
pred = model.predict(Xs_te, verbose=0).argmax(1)
print(classification_report(y_te, pred, target_names=CLASSES)); print(confusion_matrix(y_te, pred))

## 7. TFLite 변환 · 저장 · 보드로 가져갈 파일

In [ ]:
tfl = tf.lite.TFLiteConverter.from_keras_model(model).convert()
open(save_dir + 'faceshape_mlp.tflite', 'wb').write(tfl)
it = tf.lite.Interpreter(model_content=tfl); it.allocate_tensors()
i, o = it.get_input_details()[0], it.get_output_details()[0]
agree = 0
for x, y in zip(Xs_te[:200], pred[:200]):
    it.set_tensor(i['index'], x[None].astype(np.float32)); it.invoke(); agree += int(it.get_tensor(o['index'])[0].argmax() == y)
print('Keras vs TFLite 일치:', agree, '/ 200')
print()
print('보드 ~/work/FaceReading_3/ 로 복사할 파일:')
for f in ['norms.json', 'scaler.json', 'faceshape_mlp.tflite', 'wflw_mesh.tflite', 'face_detection_yunet_2023mar.onnx', 'selfie_multiclass_256x256.tflite']:
    p = save_dir + f
    print('  %-36s %s' % (f, '%.1f KB' % (os.path.getsize(p) / 1024) if os.path.exists(p) else '(없음)'))